# Model Evaluation

Consider our world. We have a trained model $\hat f$ (or, equivalently, a scoring function $\hat s$), and we want to say:

> How well does $\hat f$ perform on new data?


We might want to do this for several reasons, among them: 

- model evaulation "How good is *this* model?"
- model selection "Let me choose the *best* model."

We come up with a metric: $\hat{\text{Perf}}(D, f)$ for some dataset $D=\{(x_n,y_n)\}_{n=1}^M$ (or, equivalently, $\hat{\text{Perf}}(D, s)$) and a function $f$. This could be something like a loss we saw previously, for example:

- **Mean squared error (regression):**
$$ 
\hat{\text{Perf}}(D, f) = \frac{1}{M} \sum_{n=1}^M (y_n - f(x_n))^2
$$

- **Classification error:**
$$
\hat{\text{Perf}}(D, f) = \frac{1}{M} \sum_{n=1}^M \mathbf{1}\{f(x_n) \neq y_n\}
$$

- **Log loss (cross-entropy):**
$$
\hat{\text{Perf}}(D, s) = -\frac{1}{M} \sum_{n=1}^M \log p_s(y_n \mid x_n)
$$

But it does not have to be an average over individual points. It could also be something that depends on the dataset as a whole:

- **Correlation:**
$$
\hat{\text{Perf}}(D, f) = \hat{\mathrm{corr}}\big([y_1,\dots,y_M],\; [f(x_1),\dots,f(x_M)]\big)
$$

- **F1 score:** computed from counts of true positives, false positives, and false negatives

The key point is:

> $\hat{\text{Perf}}(D, f)$ is just a rule that takes a dataset $D$ and a model $f$, and returns a number measuring performance.

Sometimes we use losses because they are mathematically convenient, but other times we prefer metrics that are easier to interpret.

## What do we actually want to measure?

We have defined a metric $\hat{\text{Perf}}(D, f)$. The next question is:

> What data should we apply this metric to?

The answer is:

> We want to measure performance on **new, independent data**.

Isn't that the whole point of our ML world?

Assume we have a model $\hat{f}$ that has been trained from data $D_{train}$: 
$$
\hat{f} = A(D_{train})
$$
for some algorithm $A$.

We can formalize this as saying we want to estimate: 

$$
\text{Perf}(\hat f) := \mathbb{E}_{D}[\hat{\text{Perf}}(D, \hat f) \mid \hat f]
$$

Where $D$ is a newly drawn data set under the stipulation that: $D\sim P^M$ and $D \perp \hat{f}$. 

This expression is dense. We break it down:

- **$D \sim P^M$**  
  The dataset $D = \{(x_n,y_n)\}_{n=1}^M$ is a fresh sample drawn from the same distribution as our training data. Here $P^M$ is just short-hand for saying they are i.i.d.

- **$D \perp \hat f$**  
  The dataset is **independent of the model**.  
  This is what makes it “new” data. It has not been used, directly or indirectly, to construct $\hat f$. A sufficient condition for this is that $D \perp D_{train}$. (Its really about independence from $\hat{f}$ though, not from $D_{train}$. Edge case: If the model $\hat f$ is completely independent of $D_{train}$, then using the same dataset for training and evaluation is not a problem, because $D \perp \hat f$ still holds.)

- **$\hat{\text{Perf}}(D, \hat f)$**  
  We compute our chosen metric on that dataset using the trained model.

- **$\mathbb{E}[\cdot \mid \hat f]$**  
  We treat the trained model $\hat f$ as fixed, and take the expectation over all possible fresh datasets $D$.

In any case,  $\text{Perf}(\hat f)$ is the expected value of our evaluation metric when applied to new, independent datasets, holding the trained model fixed.

This captures exactly what we mean by:

> "How well does this model perform on new data?"


## What should we actually do?

In practice we have two (hopefully independent) datasets, $D_{train}$ and $D_{eval}$. We then:

1. Train:
$$
\hat f = A(D_{train})
$$

2. Evaluate:
$$
\hat{\text{Perf}}(D_{eval}, \hat f)
$$

This gives us an estimate of $\text{Perf}(\hat f)$.

When is this valid? We need two conditions:

- $D_{eval} \sim P^M$  
- $D_{eval} \perp \hat f$

If both hold, then $D_{eval}$ behaves like fresh data relative to the model, and we have some reason to believe that
$$
\hat{\text{Perf}}(D_{eval}, \hat f)
$$
is a good estimate of $\text{Perf}(\hat f)$.

If these conditions do not hold, the evaluation data may reflect how the model was constructed, and the estimate can be biased.

For example, if we evaluate on the training data ($D_{eval} = D_{train}$), then the model has already been adapted to that dataset, and the resulting estimate is typically overly optimistic. Indeed, **evaluating on the training data set typically leads to unrealistically low error estimates.**

**Some analogies for why this happens**

One way to think about it is:

> The model has already “seen” the training data.

It has been explicitly optimized to perform well on that dataset, so evaluating on it does not tell us how it behaves on anything new.

Another perspective:

> The model can exploit quirks of the dataset that do not generalize.

Even if there is real signal, there are also random patterns in any finite dataset. A flexible model can partially fit those patterns, which lowers the training error but does not reflect true performance.

Alternatively: 

> Training data tells us how well the model fits *this dataset*, not how well it generalizes.

Evaluation is about generalization, so we need data that the model has not been adapted to.

Analogy:

> It’s like studying for an exam using a set of questions, and then grading yourself on those same questions.

You will likely do very well, but that score does not reflect how you would perform on new questions. This is the **memorization** problem.

Slightly more technical intuition:

> The training process pushes the model to reduce error on that specific dataset.

So any evaluation on that dataset is "contaminated" by the optimization process itself, which systematically makes the error look smaller than it really is.

Another way to think about it:

> The model can "hallucinate" structure that is not really there.

When optimizing on a fixed dataset, the model may latch onto patterns that look meaningful in that sample but are actually accidental. These spurious patterns improve performance on the training data, but break down on new data.

## Isn't the training data informative?

Yes, but about a different quantity. Training data tells us which models fit this dataset well. What we actually care about is performance on new data. These are not completely unrelated, of course, but we have to be careful not to treat the training dataset as the be-all and end-all.

The issue is that we construct the model using the training data:
$$
\hat f = A(D_{train})
$$

So the model is adapted to that dataset. As a result it doesn't act like a new, independent set of data. It has been used to shape the model, so it cannot be treated as neutral evidence about its performance. It gives us *some* hints, but we have to be careful. 

The key is we need two things: 

- **Training data** to optimize and adapt the model  
- **Evaluation data** to measure performance without having influenced the model

## Overfitting

Overfitting is when a model fits the training data too closely, including patterns that are not part of the true underlying signal.

More concretely, the model captures:
- real structure in the data  
- but also noise and sample-specific quirks  

As a result:
- it achieves low training error  
- but performs worse on new, unseen data  

A useful way to think about it is:

> The model is not just learning the pattern, it is partially memorizing the dataset.

This is why more flexible models are more prone to overfitting: they have enough capacity to fit both signal and noise.

## Efron’s optimism theorem

In practice, training error is almost always too optimistic. The model has been tuned to perform well on that specific dataset, so it tends to look better than it really is. Making this intuition fully precise is actually quite tricky, and requires more advanced tools from statistics and learning theory. Moreover, there is no single clean result that covers all models; results typically depend on the model class and learning procedure.

Nonetheless, there is a result that helps quantify this effect in some settings, and provides useful intuition.

For a broad class of estimators (in particular, those where predictions depend smoothly on the data), one can show something like: 
$$
\mathbb{E}[\text{Perf}(\hat f) - \hat{\text{Perf}}(D_{train}, \hat f)]
= \frac{2}{N} \sum_{i=1}^N \mathrm{Cov}(\hat y_i, y_i)
$$

where:
- $\hat y_i = \hat f(x_i)$  
- the expectation is over the randomness in the responses $y$ (typically assuming fixed $x_i$)

The quantity $\text{Perf}(\hat f) - \hat{\text{Perf}}(D_{train}, \hat f)$ is often called **optimism**. It measures how much training error underestimates true error.

The key feature of this expression is:

> The gap depends on how sensitive the model’s predictions are to the data.

The more the model adapts to the data, the larger the covariance term (typically $y$ and $\hat{y}$ are positively correlated) and the more optimistic the training error becomes. What if our $\hat{y}$ are completely independent of $y$? Then we have no optimism, and we don't run into these issues.

### Concrete example: linear regression

For least squares regression with noise of variance $\sigma^2$:
$$
\mathbb{E}[\text{Perf}(\hat f) - \hat{\text{Perf}}(D_{train}, \hat f)]
= \frac{2\sigma^2}{N} \cdot d
$$

where:
- $d$ is the number of parameters  

Interpretation:

- more parameters → more flexibility  
- more flexibility → stronger dependence on the data  
- stronger dependence → larger optimism  

This result is quite general as an identity, but exact, closed-form expressions are mainly available for:

- linear regression  
- ridge / kernel methods (later)  
- other linear or smoothing estimators (later)

For more complex models (e.g., trees, ensembles), the same intuition holds, but the covariance term is difficult to compute or interpret directly.

# Some Simulations

Let's generate some independent training and evaluation data, and then fit a series of polynomial regression models:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def make_data(n: int, rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """Generate synthetic 1D regression data."""
    x = rng.uniform(-1.0, 1.0, size=n)
    y_true = np.sin(2 * np.pi * x)
    y = y_true + rng.normal(0.0, 0.25, size=n)
    return x, y


def fit_and_eval_polynomials(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    max_degree: int = 20,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Fit polynomial regression models of degrees 0..max_degree and return MSEs."""
    degrees = np.arange(max_degree + 1)
    train_mse = np.zeros_like(degrees, dtype=float)
    test_mse = np.zeros_like(degrees, dtype=float)

    for i, degree in enumerate(degrees):
        # np.polyfit solves least squares for polynomial coefficients
        coeffs = np.polyfit(x_train, y_train, deg=degree)

        yhat_train = np.polyval(coeffs, x_train)
        yhat_test = np.polyval(coeffs, x_test)

        train_mse[i] = np.mean((y_train - yhat_train) ** 2)
        test_mse[i] = np.mean((y_test - yhat_test) ** 2)

    return degrees, train_mse, test_mse

In [ ]:
# Generate train/test datasets
x_train, y_train = make_data(n=30, rng=rng)
x_test, y_test = make_data(n=1000, rng=rng)

degrees, train_mse, test_mse = fit_and_eval_polynomials(
    x_train, y_train, x_test, y_test, max_degree=15
)

# Plot train and test error versus polynomial degree
plt.figure(figsize=(8, 5))
plt.plot(degrees, np.log10(train_mse), marker="o", label="Training MSE")
plt.plot(degrees, np.log10(test_mse), marker="o", label="Testing MSE")
plt.xlabel("Polynomial degree")
plt.ylabel("log10(Mean squared error)")
plt.title("Polynomial regression: training vs testing error")
plt.xticks(degrees)
plt.legend()
plt.tight_layout()
plt.show()

Features: 

- Training MSE typically less than testing (but not always, there is some randomess)
- for small polynomial, we don't fit too hard ($D_{train}$ is still approx indep of $\hat{f}$) so the gap isn't that large
- as we increase the polynomial degree, he gap starts to get big.

Let's do it a bunch of times and summarize the results:

In [ ]:
def monte_carlo_polynomial_eval(
    n_reps: int = 200,
    n_train: int = 30,
    n_test: int = 1000,
    max_degree: int = 15,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Run repeated train/test experiments and collect MSE curves."""
    rng = np.random.default_rng(seed)

    all_train = []
    all_test = []

    for _ in range(n_reps):
        x_train, y_train = make_data(n=n_train, rng=rng)
        x_test, y_test = make_data(n=n_test, rng=rng)

        degrees, train_mse, test_mse = fit_and_eval_polynomials(
            x_train, y_train, x_test, y_test, max_degree=max_degree
        )

        all_train.append(train_mse)
        all_test.append(test_mse)

    return degrees, np.vstack(all_train), np.vstack(all_test)


def summarize_curves(curves: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return q25, median, q75 across Monte Carlo repetitions."""
    q25 = np.log10(np.percentile(curves, 25, axis=0))
    med = np.log10(np.percentile(curves, 50, axis=0))
    q75 = np.log10(np.percentile(curves, 75, axis=0))
    return q25, med, q75

In [ ]:
degrees, all_train, all_test = monte_carlo_polynomial_eval(
    n_reps=200,
    n_train=30,
    n_test=1000,
    max_degree=15,
    seed=0,
)

train_q25, train_med, train_q75 = summarize_curves(all_train)
test_q25, test_med, test_q75 = summarize_curves(all_test)

plt.figure(figsize=(9, 5))

plt.plot(degrees, train_med, label="Training MSE (median)")
plt.fill_between(degrees, train_q25, train_q75, alpha=0.2)

plt.plot(degrees, test_med, label="Testing MSE (median)")
plt.fill_between(degrees, test_q25, test_q75, alpha=0.2)

plt.xlabel("Polynomial degree")
plt.ylabel("Mean squared error")
plt.title("Polynomial regression: Monte Carlo train/test error")
plt.xticks(degrees)
plt.legend()
plt.tight_layout()
plt.show()

The general cartoon picture looks something like this: 

![Evaluation and training error versus model complexity.](https://i.sstatic.net/rpqa6.jpg)

### Why does overfitting get worse as model complexity increases?

There are a few complementary ways to understand this.

**Heuristic:**

As model complexity increases, the model has more capacity:

- it can fit the true signal better  
- but it can also fit noise and accidental patterns  

At high complexity the model is very flexible → it can match almost any dataset, including patterns that will not generalize. 

**Dependence**

Recall what we need for evaluation:

$$
D \perp \hat f
$$

Overfitting can be understood as the model becoming increasingly dependent on the specific dataset it was trained on.

As complexity increases:
- the training procedure can extract more detailed structure from the dataset  
- including noise and sample-specific quirks  

So $\hat f$ depends more strongly on $D_{train}$ and so it becomes less of a reasonable way to evaluate the method. 

**Sensitivity**

Another way to say this complex models are more sensitive to the data. Small changes in the dataset can lead to large changes in the fitted model.

- low complexity → stable, similar fits across datasets  
- high complexity → highly variable fits  

This increased sensitivity is exactly what shows up in Efron’s result: higher covariance between predictions and data, and therefore larger optimism (overfitting). 

**Search Space**

A complex model is effectively searching over a much larger space of functions.

You can think of it as trying many possible explanations and picking the one that fits best.

With more possibilities:
- there is a higher chance of finding a function that fits the dataset extremely well  
- even if that fit is partly due to noise or non-generalizable shapes.


# How can we deal with this?

In practice, we want to estimate performance. There are several ways to do this: 


**1. Train/Eval Split** Split the data into two parts: 

- $p\%$ goes into $D_{train}$
- $(1-p)\%$ goes into $D_{eval}$

Then we train on $D_{train}$ and evaluate on $D_{eval}$ to calculate $\hat{\text{Perf}}(D_{eval}, \hat{f})$. We can choose different values for $p$ e.g. $0.5$, %0.95$, ...

This is always a *little* pessimistic since we only use $p$ percent for training.

**2. Cross-validation** Instead of a single split, we repeatedly split the data into training and evaluation sets. For example, in $k$-fold cross-validation:

- Partition the dataset $D$ into $k$ disjoint subsets (folds): $D_1, \dots, D_k$
- For each $j = 1, \dots, k$:
  - define $D_{eval}^{(j)} = D_j$
  - define $D_{train}^{(j)} = D \setminus D_j$ (i.e. everything else)
  - train $\hat f^{(j)} = A(D_{train}^{(j)})$
  - compute $\hat{\text{Perf}}(D_{eval}^{(j)}, \hat f^{(j)})$

- Return a summary, e.g. the average:
$$
\frac{1}{k} \sum_{j=1}^k \hat{\text{Perf}}(D_{eval}^{(j)}, \hat f^{(j)})
$$

This uses more of the data for both training and evaluation, and typically gives a more stable estimate of performance. It is still somewhat pessimistic since it only uses $(k-1)/k$ percentage of the data to train in each fold.

If we do this with $k=N-1$ (where $N$ is the size of our data) this is known as **leave-one-out (LOO) cross validation**.

**3. Resampling (repeated random splits)** Instead of a fixed partition, we repeatedly draw random train/evaluation splits (without replacement).

- For $b = 1, \dots, B$:
  - randomly split $D$ into $D_{train}^{(b)}$ and $D_{eval}^{(b)}$ (say $p$ percent to $D_{train}^{(b)}$)
  - train $\hat f^{(b)} = A(D_{train}^{(b)})$
  - compute $\hat{\text{Perf}}(D_{eval}^{(b)}, \hat f^{(b)})$

- Return a summary, e.g., the average:
$$
\frac{1}{B} \sum_{b=1}^B \hat{\text{Perf}}(D_{eval}^{(b)}, \hat f^{(b)})
$$

Each split produces a slightly different estimate, and averaging over many such splits reduces variability and gives a more stable estimate of performance. Resampling is a bit better at estimating variability than cross-validation, but a little less systematic.  

## What model do we actually use?

A common question is:

> If we train $k$ models in cross-validation, which one do we actually use?

The answer is:

> None of them.

Cross-validation (and similar procedures) are used for **estimating performance**, not for producing the final model.

Once we have estimated performance, we then train a **final model on all available data**:
$$
\hat f = A(D_{all})
$$

Why does this make sense? At first, this can feel a bit strange. We estimate performance using models trained on subsets of the data, but then deploy a model trained on the full dataset. This highlights this fact: these procedures are not estimating the performance of one fixed model.

Recall that for a **fixed model** $f$, we defined:
$$
\text{Perf}(f) := \mathbb{E}_{D}[\hat{\text{Perf}}(D, f) \mid f]
$$

This is the expected performance of a fixed model when evaluated on new, independent data.

Now consider what happens in practice. The model is not fixed:
$$
\hat f = A(D)
$$

So $\hat f$ depends on the training data $D$ and is therefore random. Something like cross validation is essentially estimating
$$
\mathbb{E}_{D \sim P^N}[\text{Perf}(A(D))]
$$
**This algorithm A should be construed broadly as whatever produces our final estimator.**

Equivalently, since $\hat f$ depends on $D$, we can write:
$$
\mathbb{E}_{\hat{f}}[\text{Perf}(\hat f)]
$$

This is the expected value of $\text{Perf}(\hat f)$ over the randomness in the training data (i.e., averaging over the different models $\hat f$ that the procedure could produce). 

# Code Example:

**Basic train/eval split**:

Let's go back to our **mnist** dataset and do a simple test/train split:

In [ ]:
from sklearn.datasets import fetch_openml
X, y = fetch_openml("mnist_784", version=1, as_frame=False, return_X_y=True)
y = y.astype(int)
X = X / 255.0 #rescale to 0-1

# subset
n = 2000
X = X[:n]
y = y[:n]

In [ ]:
X.shape

In [ ]:
y.shape

Let's assume that `X` and `y` have *all* of our data. We can do a "test/train" (i.e. eval/train) split using the function `train_test_split`:

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=8789687
)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
X_test.shape

In [ ]:
y_test.shape

Now we fit the model:

In [ ]:
from sklearn.linear_model import LogisticRegression
mod = LogisticRegression()
mod.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
y_pred = mod.predict(X_test)

In [ ]:
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print("Accuracy:", acc)
print("F1:", f1)

**Cross validation**

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=7987564)

for train_idx, eval_idx in kf.split(X):
    print(train_idx[(-10):])
    print(eval_idx[(-10):])
    print("\n")

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=7987564)

acc = []

for train_idx, eval_idx in kf.split(X):
    X_train, X_eval = X[train_idx], X[eval_idx]
    y_train, y_eval = y[train_idx], y[eval_idx]
    mod = LogisticRegression()
    mod.fit(X_train, y_train)
    y_pred = mod.predict(X_eval)
    acc.append(accuracy_score(y_eval, y_pred))

In [ ]:
acc

In [ ]:
np.median(acc)

In [ ]:
np.std(acc)

**Sklearn built in cross-val**

This will generally work as long as we have a `.fit` and `.predict` method defined for the object passed in.

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
mod = LogisticRegression()
scores = cross_val_score(mod, X, y, cv=5, scoring='accuracy')

In [ ]:
print("Scores:", scores)
print("Mean accuracy:", scores.mean())
print("Std accuracy:", scores.std())